# **Лабораторная работа** №5

## **Тема: Приложение для анализа заказов интернет-магазина**

## **Цель работы**

Разработать приложение, выполняющее анализ заказов и формирующее персонализированный отчёт с использованием различных библиотек Python.






## **Задание**

Создать программу, работающую по следующему сценарию:


### **1. Ввод и обработка данных пользователя**

Запросить дату рождения пользователя (формат YYYY-MM-DD).

Использовать:

* `datetime` — для получения текущей даты и вычисления возраста в днях;
* `calendar` — для определения дня недели рождения;
* `re` — для проверки корректности формата ввода.

Результат:

* возраст в днях;
* день недели (в текстовом виде).

👉 Эти данные используются в итоговом отчёте.


### **2. Загрузка и подготовка данных заказов**

Загрузить файл `orders.csv`.

Использовать:

* `pandas` — для загрузки данных (`read_csv`), обработки таблицы и преобразования столбца с датой (`to_datetime`).

Выполнить:

* проверку типов данных;
* обработку пропущенных значений;
* удаление некорректных записей.

👉 Подготовленные данные используются далее для анализа.


### **3. Расчёт показателей**

Добавить новые столбцы:

* общая стоимость заказа;
* итоговая стоимость с учётом скидки;
* признак успешной продажи.

Использовать:

* `pandas` — для вычислений по столбцам DataFrame.

👉 Эти значения являются основой для аналитики.


### **4. Анализ данных**

Выполнить расчёты:

* общий доход;
* доход по категориям;
* топ товаров;
* уровень возвратов.

Использовать:

* `pandas` — группировка (`groupby`), агрегации (`sum`, `mean`, `count`).

👉 Результаты включаются в итоговый отчёт.


### **5. Мат расчёт**

Реализовать вычисление, например:

* оценка времени доставки на основе расстояния.
* ... (по 3 индивидуальных расчета на ваше усмотрение)

Использовать:

* `math` — математические функции (например, `sqrt`);
* `math.isnan()` — проверка корректности числового ввода.

👉 Результат включается в отчёт.


### **6. Формирование отчёта**

Сформировать текстовый отчёт, содержащий:

* данные пользователя;
* результаты анализа;
* дополнительные вычисления.

Использовать:

* `pandas` — для сохранения результатов в CSV (`to_csv`);
* стандартные средства Python — для формирования текста отчёта.

👉 Отчёт должен быть сохранён в файл.


### **7. Озвучивание отчёта**

Преобразовать текст отчёта в аудио (.mp3).

Использовать:

* `gTTS` — генерация речи из текста.

Проверить:

* что текст не пустой;
* корректность выбранного языка.

### **8. Интерфейс пользователя**

Создать интерфейс для работы с программой.

Использовать:

* `tkinter` — для создания графического интерфейса (ввод данных, кнопки, вывод результатов).

👉 Интерфейс объединяет все функции приложения.


## **Требования**

* использовать библиотеки: `datetime`, `calendar`, `re`, `math`, `pandas`, `tkinter`, `gTTS`;
* обеспечить обработку ошибок (`try/except`);


## **Результат**

Программа, которая:

* получает данные пользователя;
* анализирует заказы интернет-магазина;
* выполняет вычисления;
* формирует текстовый и аудио-отчёт через интерфейс.


In [2]:
from tkinter import * # создает окна
import datetime
import calendar
import re
import math
import pandas as pd # работает с таблицами и CSV файлами
from gtts import gTTS # текст в голос

def analyze():
    try:
        birth_date = entry.get()

        # проверка формата
        if not re.match(r"\d{4}-\d{2}-\d{2}", birth_date):
            result_label.config(text="Неверный формат даты!")
            return

        # работаю с датами
        birth = datetime.datetime.strptime(birth_date, "%Y-%m-%d")
        today = datetime.datetime.now()

        age_days = (today - birth).days
        weekday = calendar.day_name[birth.weekday()]

        # загрузка CSV
        df = pd.read_csv("orders.csv")

        # очищаю данные
        df = df.dropna()
        df["date"] = pd.to_datetime(df["date"], errors='coerce')
        df = df.dropna()

        # расчёты
        df["total"] = df["price"] * df["quantity"]
        df["final"] = df["total"] * (1 - df["discount"])
        df["success"] = df["status"] == "delivered"

        total_income = df["final"].sum()
        by_category = df.groupby("category")["final"].sum()

        top_products = df.groupby("product")["quantity"].sum().sort_values(ascending=False).head(3)

        returns = df[df["status"] == "returned"].shape[0] / len(df)

        # мат. расчёты
        distance = 100
        speed = 50

        delivery_time = distance / speed
        sqrt_example = math.sqrt(distance)
        check_nan = math.isnan(delivery_time)

        # сам отчет
        report = f"""
Возраст (дни): {age_days}
День рождения: {weekday}

Общий доход: {total_income}

Доход по категориям:
{by_category}

Топ товары:
{top_products}

Уровень возвратов: {returns}

Время доставки: {delivery_time}
Корень из расстояния: {sqrt_example}
Проверка NaN: {check_nan}
"""

        # сохранение отчёта
        with open("report.txt", "w", encoding="utf-8") as f:
            f.write(report)

        # сохранение CSV
        df.to_csv("processed_orders.csv", index=False)

        # озвучка
        if report.strip() != "":
            tts = gTTS(report, lang="ru")
            tts.save("report.mp3")

        result_label.config(text="Готово! Проверь файлы.")

    except Exception as e:
        result_label.config(text=f"Ошибка: {e}")


# интерфейс
root = Tk()
root.title("Анализ заказов")

Label(root, text="Введите дату рождения (YYYY-MM-DD)").pack()

entry = Entry(root)
entry.pack()

Button(root, text="Анализировать", command=analyze).pack()

result_label = Label(root, text="")
result_label.pack()

root.mainloop()